# Slice 6 DistilBERT on private Kaggle
Run setup before changing the mode. Use a compatible GPU runtime. Keep the research input and result archive private. The result archive is needed to resume in a new session.


In [ ]:
MODE = "setup"  # setup, pilot, screen, repeat
CAMPAIGN = ""
CODE_MANIFEST_SHA256 = ""
RESEARCH_RECEIPT_SHA256 = ""
PRIVATE_INPUTS_AND_OUTPUTS_VERIFIED = False
GPU_HOURS_LEFT = None
SESSION_HOURS_LEFT = None
SEGMENT_HOURS = 2
PHYSICAL_BATCH = 8
EXECUTION = "auto"  # auto or serial
RESTORE = False
RETURN_INDEX_SHA256 = ""


In [ ]:
from hashlib import sha256
import json
from pathlib import Path, PurePosixPath
import runpy
import shutil
import stat
import subprocess
import sys
import tempfile
from zipfile import ZipFile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")

def one_input(name):
    candidates = list(INPUT.rglob(name))
    if len(candidates) != 1:
        raise ValueError("Attach exactly one private input containing " + name)
    return candidates[0]

archive_path = one_input("slice-6-code.bin")
with ZipFile(archive_path) as archive:
    names = archive.namelist()
    if len(set(names)) != len(names) or "handoff-manifest.json" not in names:
        raise ValueError("Malformed code handoff.")
    manifest_bytes = archive.read("handoff-manifest.json")
    if sha256(manifest_bytes).hexdigest() != CODE_MANIFEST_SHA256:
        raise ValueError("Code manifest digest differs from the local handoff record.")
    manifest = json.loads(manifest_bytes)
    if manifest["campaign_kind"] != "slice-6-code" or set(names) != {*manifest["files"], "handoff-manifest.json"}:
        raise ValueError("Unexpected code handoff inventory.")
    bundle = WORK / ("slice6-code-" + CODE_MANIFEST_SHA256[:12])
    with tempfile.TemporaryDirectory(prefix=".slice6-code-", dir=WORK) as temporary:
        staged = Path(temporary) / "bundle"
        staged.mkdir()
        for item in archive.infolist():
            parts = PurePosixPath(item.filename)
            if (parts.is_absolute() or ".." in parts.parts or parts.as_posix() != item.filename
                    or stat.S_IFMT(item.external_attr >> 16) not in (0, stat.S_IFREG)):
                raise ValueError("Invalid archive path or member type.")
            data = archive.read(item)
            if item.filename in manifest["files"] and sha256(data).hexdigest() != manifest["files"][item.filename]:
                raise ValueError("Code archive failed digest verification.")
            destination = staged / item.filename
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(data)
        if not bundle.exists():
            staged.rename(bundle)
        elif any(not (bundle / name).is_file() or sha256((bundle / name).read_bytes()).hexdigest() != digest
                 for name, digest in manifest["files"].items()):
            raise ValueError("Existing code handoff content differs.")

boot = runpy.run_path(str(bundle / "code/scripts/kaggle_bootstrap.py"))
code_root, _ = boot["prepare_handoff"](bundle, WORK)
closure = boot["dependency_closure"]()
evidence = WORK / "artifacts" / "slice-6-runtime"
evidence.mkdir(parents=True, exist_ok=True)
(evidence / "dependency-closure.json").write_text(json.dumps(closure, indent=2, sort_keys=True))
print("Verified installed dependency closure:", len(closure), "distributions")
print("Free workspace GiB:", round(shutil.disk_usage(WORK).free / 2**30, 2))
subprocess.run([sys.executable, "-m", "disaster_triage", "doctor",
                "--root", str(WORK), "--device", "cuda:0"], check=True)


In [ ]:
if MODE not in ("setup", "pilot", "screen", "repeat"):
    raise ValueError("Choose setup, pilot, screen, or repeat.")
if MODE != "setup":
    if not PRIVATE_INPUTS_AND_OUTPUTS_VERIFIED:
        raise ValueError("Verify private inputs, notebook, collaborators and outputs before training.")
    if not CAMPAIGN or not RESEARCH_RECEIPT_SHA256:
        raise ValueError("Enter the campaign ID and research receipt digest from local preparation.")
    argv = [sys.executable, "-m", "disaster_triage", "transformer", "run",
            "--root", str(WORK), "--source-root", str(code_root), "--campaign-id", CAMPAIGN,
            "--input", str(one_input("slice-6-research.bin")),
            "--receipt-sha256", RESEARCH_RECEIPT_SHA256, "--stage", MODE, "--backend", "cuda",
            "--execution", EXECUTION, "--batch", str(PHYSICAL_BATCH), "--privacy-confirmed",
            "--segment-hours", str(SEGMENT_HOURS)]
    for option, value in (("--gpu-hours-left", GPU_HOURS_LEFT),
                          ("--session-hours-left", SESSION_HOURS_LEFT)):
        if value is not None:
            argv.extend([option, str(value)])
    if RESTORE:
        if not RETURN_INDEX_SHA256:
            raise ValueError("Enter the return-index digest from the prior session.")
        argv.extend(["--results", str(one_input("slice-6-results.bin")),
                     "--return-index-sha256", RETURN_INDEX_SHA256])
    output = subprocess.check_output(argv, text=True)
    result = json.loads(output)
    print(json.dumps(result, indent=2))
    from IPython.display import FileLink, display
    display(FileLink(str(Path(result["result_archive"]).relative_to(WORK))))
